In [1]:
%matplotlib inline

import sys
import matplotlib.pyplot as plt

sys.path.append('../../../')

In [2]:
from __future__ import annotations

import glob
import math
import yaml
import os
import random
from copy import deepcopy
from pathlib import Path
from typing import Any

import cv2
import numpy as np
from torch.utils.data import Dataset

%load_ext autoreload
%autoreload 2
    
from computer_vision.yolov11_pose.utils import DEFAULT_CFG_DICT
from computer_vision.yolov11_pose.data.utils import IMG_FORMATS, img2label_paths, exif_size, verify_image_label

class YOLODataset(Dataset):
    """Base dataset class for loading and processing image data

    This class provides core functionality for loading images, caching, and preparing data for training and inference
    """
    def __init__(self, data:dict|None=None, task:str='detect', img_path:str|list[str]=None, imgsz:int=640, 
                 cache:bool|str=False,augment:bool=True, hyp:dict[str, any]=DEFAULT_CFG_DICT,
                 prefix:str='',rect:bool=False,batch_size:int=16,stride:int=32,pad:float=0.5,
                 single_cls:bool=False,classes:list[int]|None=None,
                 fraction:float=1.,channels:int=3):
        """Initialize YOLODataset with given configuration and options
        Args:
            data (dict|None): Dataset configuration dictionary
            task (str): Task type, one of `detect`, `segment`, `pose`, or `oob`
            img_path (str|list[str]): Path to the folder containing images or list of image paths
            imgsz (int): Image size for resizing
            cache (bool): Cache images to RAM or disk during training
            augment (bool): If True, data augmentation is applied
            hyp (dict[str, Any]): Hyperparameters to apply data augmentation
            prefix (str): Prefix to print log messages
            rect (bool): If True, rectangular training is used
            batch_size (int): Size of batches
            stride (int): Stride used in the model
            pad (float): Padding value
            single_cls (bool): If True, single class training is used
            classes (list[int],optional): List of included classes 
            fraction (float): Fraction of dataset to utilize
            channels (int): Number of channels in the images ( 1 for grayscale and 3 for RGB)
        """
        super().__init__()
        self.use_segments=task=='segment'
        self.use_keypoints=task=='pose'
        self.use_obb=task=='obb'
        if isinstance(data, dict): self.data=data
        elif isinstance(data, str): data=Path(data)
        if isinstance(data, Path):
            assert data.is_file(), f'{data} does not exist'
            with open(data, encoding="utf8") as f: self.data=yaml.load(f, Loader=yaml.SafeLoader)
        
        self.img_path=img_path
        self.imgsz=imgsz
        self.augment=augment
        self.single_cls=single_cls
        self.prefix=prefix
        self.fraction=fraction
        self.channels=channels
        self.cv2_flag=cv2.IMREAD_GRAYSCALE if channels==1 else cv2.IMREAD_COLOR
        self.im_files=self.get_img_files(self.img_path)

    def get_img_files(self, img_path:str|list[str])->list[str]:
        """Read image files from the specified path
        Args:
            img_path (str|list[str]): Path or list of paths to image directories or files
        Returns:
            (list[str]): List of image file paths
        """
        f=[] # image files
        for p in img_path if isinstance(img_path, list) else [img_path]:
            p=Path(p) # os agnosic
            if p.is_dir(): 
                f+=glob.glob(str(p/"**"/"*.*"), recursive=True)
            elif p.is_file():
                with open(p, encoding='utf-8') as t:
                    t=t.read().strip().splitlines()
                    parent=str(p.parent)+os.sep
                    f+=[x.replace('./', parent) if x.startwith('./') else x for x in t] # local to global path
            else: raise FileNotFoundError(f'{self.prefix}{p} does not exist')
        im_files=sorted(x.replace('/',os.sep) for x in f if x.rpartition('.')[-1].lower() in IMG_FORMATS)
        assert im_files, f'{self.prefix}No images found in {img_path}'
        if self.fraction<1:
            im_files=im_files[:round(len(im_files)*self.fraction)] # retain a fraction of the dataset
        return im_files

    def get_labels(self)->list[dict]:
        """Return dict of labels for YOLO training

        This method loads labels from disk or cache, verifies their integrity, and prepares them for training
        
        Returns:
            (list[dict]): List of label dict, each containing information about an image and its annotations
        """
        self.label_files=img2label_paths(self.im_files)
        cache_path=Path(self.label_files[0]).parent.with_suffix(".cache")
        
        

data_dirpath='D:/data/ultralytics/coco8-pose'
dataset=YOLODataset(data='../coco8-pose.yaml', task='pose', img_path=data_dirpath, imgsz=640, cache=False,augment=True, hyp=DEFAULT_CFG_DICT,
                prefix='',rect=False,batch_size=16,stride=32,pad=0.5, single_cls=False,classes=None,
                fraction=1.,channels=3)

In [3]:
# def get_labels

dataset.label_files=img2label_paths(dataset.im_files)
cache_path=Path(dataset.label_files[0]).parent.with_suffix(".cache")



In [4]:
path = cache_path
#def cache_labels
x={'labels':[]}
nm, nf, ne, nc, msgs=0,0,0,0,[] # number of missing, found, empty, corrupt, messages
desc=f'{dataset.prefix}Scanning {path.parent/path.stem}...'
total=len(dataset.im_files)
nkpt,ndim=dataset.data.get('kpt_shape', (0,0))
if dataset.use_keypoints and (nkpt<=0 or ndim not in {2,3}):
    raise ValueError("'kpt_shape' in data.yaml missing or incorrect, Should be a list with [number of keypoints,"
                     "number of dims (2 for x,y or 3 for x,y,visible)], i.e.,'kpt_shape:[17,3]'")
print(f'nkpt {nkpt},ndim {ndim}')

nkpt 17,ndim 3


In [8]:
import cv2
import numpy as np
from PIL import Image, ImageOps

from computer_vision.yolov11_pose.utils.ops import segments2boxes

# def verify_image_label
im_file=dataset.im_files[0]
lb_file=dataset.label_files[0]
prefix=dataset.prefix
keypoint=dataset.use_keypoints
num_cls=len(dataset.data['names'])
#nkpt
#ndim
single_cls=dataset.single_cls

im_file, lb, shape, segments, keypoints, nm, nf, ne, nc, msg=verify_image_label(im_file, lb_file, prefix, keypoint, num_cls, nkpt, ndim, single_cls)

In [11]:
im_file

'D:\\data\\ultralytics\\coco8-pose\\images\\train\\000000000036.jpg'

In [10]:
lb.shape

(1, 5)